# OpenPlaque — Plaque + Inflammation Best Estimates v1.1

This is the corrected consolidated research endpoint for **LAD, RCA, LCX, and left main (LM)**.

It aligns the presentation with:
- van Rosendael et al., CONFIRM2 2026 — quantitative TPV/NCPV staging;
- Shaw et al., SCCT/NASCI 2021 — structured plaque reporting/high-risk plaque terminology;
- Chan et al., ORFAN 2024 — three-vessel inflammatory-risk framework;
- Oikonomou et al. 2021 — standardized coronary inflammation / FAI framework.

**Important:** these are OpenPlaque research estimates, not Cleerly or Caristo outputs. LM 54 mm³ is a calcium-volume anchor only; LM NCPV/LAP are therefore NA rather than zero. Aggregate NCPV/LAP are known lower estimates from LAD/RCA/LCX.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Paths / reuse controls — immediately after Drive mount
DRIVE_ROOT = "/content/drive/MyDrive/OpenPlaque"
OUT = f"{DRIVE_ROOT}/Plaque_Inflammation_Best_Estimates_v1_1"

REQUIRED = [
    f"{DRIVE_ROOT}/UCLA_Plaque_Type_Estimates/best_estimate_plaque_types_by_artery.csv",
    f"{DRIVE_ROOT}/RCA_Plaque_PCAT_Research_Lock_v1/RCA_locked_research_plaque_PCAT_profile_10_50.csv",
    f"{DRIVE_ROOT}/LAD_Source_Space_PCAT_Feasibility_v1/LAD_PCAT_segment_summary.csv",
    f"{DRIVE_ROOT}/LCX_OM_Source_Space_Composition_PCAT_Feasibility_v1/LCX_OM_PCAT_primary_summary.csv",
]

from pathlib import Path
for p in REQUIRED:
    if not Path(p).is_file():
        raise FileNotFoundError(p)

print("Output:", OUT)
print("All cached evidence inputs found.")

In [ ]:
import shutil, sys, subprocess, json
from pathlib import Path

OPENPLAQUE_PIN = "efe756b9931186cf724c7ff1f305fc43bf5335b3"
OPENPLAQUE_BRANCH = "plaque-inflammation-best-estimates-from-main"

if Path("/content/OpenPlaque").exists():
    shutil.rmtree("/content/OpenPlaque")

!git clone -q --branch {OPENPLAQUE_BRANCH} https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
!git -C /content/OpenPlaque checkout -q {OPENPLAQUE_PIN}

%pip install -q /content/OpenPlaque

for name in list(sys.modules):
    if name == "openplaque" or name.startswith("openplaque."):
        del sys.modules[name]

print("OpenPlaque pin:", subprocess.check_output(["git","-C","/content/OpenPlaque","rev-parse","HEAD"], text=True).strip())

In [ ]:
# Synthetic tests before study-data fusion
from openplaque.plaque_inflammation_best_estimates_v1 import synthetic_self_test
print(synthetic_self_test())
!cd /content/OpenPlaque && pytest -q tests/test_plaque_inflammation_best_estimates_v1.py

In [ ]:
from openplaque.plaque_inflammation_best_estimates_v1 import run

summary = run(
    drive_root=DRIVE_ROOT,
    output_dir=OUT,
)

print(json.dumps({
    "status": summary["status"],
    "major_vessel_TPV_best_estimate_mm3": summary["headline_major_vessel_tpv_best_estimate_mm3"],
    "CONFIRM2_TPV_stage": summary["headline_confirm2_tpv_stage"],
}, indent=2))

In [ ]:
# Plaque estimates — Cleerly/CONFIRM2-style presentation
import pandas as pd
from IPython.display import display

plaque = pd.read_csv(Path(OUT)/"plaque_best_estimates_by_vessel.csv")
whole = pd.read_csv(Path(OUT)/"major_vessel_aggregate.csv")

cols = [
    "vessel",
    "tpv_best_estimate_mm3",
    "tpv_strict_or_known_lower_mm3",
    "tpv_candidate_envelope_upper_mm3",
    "ncpv_best_estimate_mm3",
    "lap_best_estimate_mm3",
    "calcified_plaque_volume_mm3",
    "confirm2_tpv_stage",
    "confirm2_ncpv_stage",
    "absolute_volume_confidence",
]
display(plaque[cols])
display(whole)

print("""
TPV = total plaque volume proxy.
NCPV = all OpenPlaque plaque bins below 350 HU.
LAP = -30 to <30 HU proxy from the legacy plaque classifier.
PAV is intentionally NA because a defensible source-space outer-vessel volume
is not available for every territory.
""")

In [ ]:
# Inflammation — Caristo-comparable raw PCAT, not proprietary FAI-Score
infl = pd.read_csv(Path(OUT)/"inflammation_best_estimates_by_vessel.csv")

display(infl[[
    "vessel",
    "pcat_mean_hu_best_estimate",
    "pcat_segment_length_mm",
    "standard_target_length_mm",
    "segment_coverage_fraction",
    "caristo_comparability",
    "fai_score",
    "confidence",
]])

print("""
Caristo/ORFAN FAI-Score is NOT reconstructed here.
Raw PCAT attenuation is reported because it is directly measurable.
The proprietary standardization for scan parameters, anatomy/biology, age and sex
is not available to OpenPlaque.
""")

In [ ]:
# Figures + report
from IPython.display import Image, display, HTML

for name in [
    "01_plaque_composition_best_estimate.png",
    "02_plaque_uncertainty.png",
    "03_inflammation_raw_pcat.png",
]:
    print(name)
    display(Image(filename=str(Path(OUT)/name), width=950))

report = Path(OUT)/"OPENPLAQUE_PLAQUE_INFLAMMATION_BEST_ESTIMATES_V1_REPORT.html"
display(HTML(report.read_text()))

In [ ]:
# Literature alignment used by this endpoint
lit = pd.read_csv(Path(OUT)/"literature_alignment.csv")
display(lit)

In [ ]:
# Verify deliverables
expected = [
    "run_state.json",
    "summary.json",
    "plaque_best_estimates_by_vessel.csv",
    "major_vessel_aggregate.csv",
    "inflammation_best_estimates_by_vessel.csv",
    "literature_alignment.csv",
    "01_plaque_composition_best_estimate.png",
    "02_plaque_uncertainty.png",
    "03_inflammation_raw_pcat.png",
    "OPENPLAQUE_PLAQUE_INFLAMMATION_BEST_ESTIMATES_V1_REPORT.html",
    "OPENPLAQUE_PLAQUE_INFLAMMATION_BEST_ESTIMATES_V1_RESULTS.zip",
]
missing=[x for x in expected if not (Path(OUT)/x).exists()]
if missing:
    raise RuntimeError("Missing outputs: "+str(missing))

state=json.loads((Path(OUT)/"run_state.json").read_text())
if state.get("status")!="COMPLETE":
    raise RuntimeError("Run state not COMPLETE: "+str(state))

print("COMPLETE")
for x in expected:
    print(Path(OUT)/x)